# TATR / TMTR — Reduced Replication
## FTS-Diffusion Paper, Section 5.3 / Fig. 6

### Multi-asset, multi-experiment notebook
Supports: **S&P 500**, **GOOG**, **ZC=F** for both **TATR** and **TMTR** experiments.

### Configuration (reduced from paper to fit Colab Pro A100)
- **15 runs** (paper: 100) — for 95% CI estimate
- **101 augmentation steps × 252 days = 100 years** of synthetic data per run (matches paper)
- **Train LSTM at 11 evaluation points**: 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100 years
- **Total LSTMs trained**: 15 × 11 = **165** per asset (paper: ~10,100)

### Persistent storage on Drive
```
/content/drive/MyDrive/fts_diffusion/
├── architectures/{asset}/   # Pre-trained FTS-Diffusion per asset
├── synthetic/{asset}/       # Generated synthetic series per run
├── results/{exp}/{asset}/   # MAPE CSVs + summary
└── figures/{exp}/           # Final plots
```

### Paper-faithful hyperparameters
| Param | Value |
|-------|-------|
| LSTM hidden_dim | 32 |
| LSTM layers | 2 |
| Window size | 64 |
| Loss | MAE |
| Optimizer | Adam, lr=1e-2 |
| Epochs | 100 |
| Training mode | **Full-batch** |

## 1. Environment Setup

In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    if not os.path.exists(CLONE_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])

    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'
    PERSIST_ROOT = '/content/drive/MyDrive/fts_diffusion'
else:
    here = Path.cwd().resolve()
    repo_candidates = [here / 'fts-diffusion-ref', here.parent / 'fts-diffusion-ref']
    try:
        REPO_ROOT = str(next(p for p in repo_candidates if p.exists()))
    except StopIteration as exc:
        raise RuntimeError(f'Could not find fts-diffusion-ref from cwd={here}') from exc
    PERSIST_ROOT = str(Path(REPO_ROOT).parent / 'fts_diffusion_run')

# Create top-level Drive structure (subdirs created lazily once ASSET/EXPERIMENT chosen)
os.makedirs(PERSIST_ROOT, exist_ok=True)
for sub in ['architectures', 'synthetic', 'results', 'figures']:
    os.makedirs(f'{PERSIST_ROOT}/{sub}', exist_ok=True)

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data', 'res', 'trained_models', 'figs']:
    os.makedirs(d, exist_ok=True)

print(f"Working dir:    {os.getcwd()}")
print(f"Persistent dir: {PERSIST_ROOT}")

In [ ]:
# Install dependencies
IMPORT_NAMES = {
    'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
    'scipy': 'scipy', 'scikit-learn': 'sklearn', 'torch': 'torch',
    'tqdm': 'tqdm', 'tslearn': 'tslearn', 'dtaidistance': 'dtaidistance',
    'yfinance': 'yfinance',
}
for pkg, import_name in IMPORT_NAMES.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("\u2713 Deps ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU info
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # For maximum reproducibility (slightly slower):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("\u2713 Deterministic mode ON (TF32 disabled)")

# Authors' modules
from utils.load_data import get_fts, load_actual_fts
from models.model_params import prm_params, pgm_params, pem_params
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)
from models.sampling import generate_timeseries_ftsdiffusion
from experiments.utils_downstream import (
    setup_dowmstream_tatr, init_first_segment,
    Timeseries2Dataset_Downstream, create_syn_dataset,
    concat_datasets_downstream, copy_dataset_downstream)
from experiments.predictor_lstm import LSTMPredictor, set_loss_fn

print("\u2713 Imports ready")

## 2. Configuration

In [ ]:
# ============================================================================
# === EXPERIMENT CONFIG ===
# Change ASSET and EXPERIMENT to switch dataset / experiment type.
# All paths on Drive are derived from these two variables.
# ============================================================================

ASSET          = 'sp500'      # 'sp500' | 'goog' | 'zcf'
EXPERIMENT     = 'tatr'       # 'tatr'  | 'tmtr'

# Per-asset metadata (matches paper Appendix E.1)
ASSETS_CONFIG = {
    'sp500': {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500'},
    'goog':  {'ticker': 'GOOG',  'start': '2005-01-01', 'end': '2020-01-01', 'pretty': 'GOOG'},
    'zcf':   {'ticker': 'ZC=F',  'start': '2001-01-01', 'end': '2020-01-01', 'pretty': 'ZC=F (Corn futures)'},
}
assert ASSET in ASSETS_CONFIG, f"Unknown asset {ASSET}"
assert EXPERIMENT in ['tatr', 'tmtr'], f"Unknown experiment {EXPERIMENT}"

DATANAME       = ASSET
TICKER         = ASSETS_CONFIG[ASSET]['ticker']
START_DATE     = ASSETS_CONFIG[ASSET]['start']
END_DATE       = ASSETS_CONFIG[ASSET]['end']
PRETTY_NAME    = ASSETS_CONFIG[ASSET]['pretty']

# For faithful reproduction, S&P 500 uses the exact authors' downstream split:
# first 5 trading years of the held-out 20% block for initial LSTM training, rest for testing.
# GOOG/ZC=F have shorter held-out blocks under the reference 80/20 split, so they require
# an explicitly labeled adaptive exploratory split unless you provide author-specific settings.
TATR_INIT_MODE = 'paper' if ASSET == 'sp500' else 'adaptive'

# The author-provided supplementary TATR code uses the first observed downstream segment
# deterministically. Algorithm 2 describes generic sampling and says the initial segment is
# sampled from observed data, but the reported downstream TATR implementation fixes it.
# Keep 'reference_first' for primary author-code replication; use 'paper_sampled' only as
# a sensitivity check for the paper's higher-level sampling description.
INIT_SEGMENT_MODE = 'reference_first'  # 'reference_first' | 'paper_sampled'

# === Persistent paths derived from ASSET / EXPERIMENT ===
ARCH_DIR    = f'{PERSIST_ROOT}/architectures/{ASSET}'
SYN_DIR     = f'{PERSIST_ROOT}/synthetic/{ASSET}'
RES_DIR     = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}'
FIG_DIR     = f'{PERSIST_ROOT}/figures/{EXPERIMENT}/{ASSET}' # Plots per experiment + asset
for d in [ARCH_DIR, SYN_DIR, RES_DIR, FIG_DIR,
          f'{ARCH_DIR}/res', f'{ARCH_DIR}/trained_models', f'{ARCH_DIR}/data']:
    os.makedirs(d, exist_ok=True)

# === Override the global prm_params with the chosen asset ===
from models.model_params import prm_params
prm_params['dataname'] = ASSET

# === Experiment hyperparameters ===
N_RUNS         = 15

# EVAL_YEARS: which augmentation lengths to evaluate the LSTM at.
# This list is MUTABLE: you can add/remove years between sessions.
# - Already-computed years are loaded from disk (no recomputation).
# - New years are computed using the EXISTING saved synthetic data.
# - As long as max(EVAL_YEARS) <= SYN_MAX_YEARS, no synthetic regeneration is needed.
EVAL_YEARS     = [0, 10, 30, 50, 70, 90]

# SYN_MAX_YEARS: how much synthetic data to generate ONCE per run (independent of EVAL_YEARS).
# Make this generous (e.g. 100) so you can add new EVAL_YEARS in the future
# without regenerating synthetic data.
# WARNING: if you increase this AFTER having already saved synthetic .npy files
# with a smaller length, the runner will REGENERATE them automatically.
SYN_MAX_YEARS  = 100
SYN_CACHE_VERSION = 'blockseed_v1'

AUG_LENGTH     = 252

# Sanity: every EVAL_YEAR must be \u2264 SYN_MAX_YEARS
assert max(EVAL_YEARS) <= SYN_MAX_YEARS, (
    f"max(EVAL_YEARS)={max(EVAL_YEARS)} > SYN_MAX_YEARS={SYN_MAX_YEARS}. "
    f"Either reduce EVAL_YEARS or increase SYN_MAX_YEARS.")

# === LSTM CONFIG (paper-faithful) ===
WINDOW_SIZE    = 64
LSTM_HIDDEN    = 32
LSTM_LAYERS    = 2
AHEAD          = 1
N_EPOCHS       = 100
LR             = 1e-2
LSTM_LOSS      = 'mae'
DATATYPE       = 'prices'

# Keep result caches tied to the downstream protocol. This avoids silently reusing stale MAPE
# CSVs after changing the split, window, horizon, LSTM size, loss, epochs, or data type.
RESULT_TAG = (
    f'{TATR_INIT_MODE}_{INIT_SEGMENT_MODE}_w{WINDOW_SIZE}_a{AHEAD}_h{LSTM_HIDDEN}_'
    f'l{LSTM_LAYERS}_{LSTM_LOSS}_e{N_EPOCHS}_{DATATYPE}'
)
RES_DIR = f'{RES_DIR}/{RESULT_TAG}'
FIG_DIR = f'{FIG_DIR}/{RESULT_TAG}'
os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  ASSET         = {ASSET}  ({PRETTY_NAME})")
print(f"  EXPERIMENT    = {EXPERIMENT.upper()}")
print(f"  N_RUNS        = {N_RUNS}")
print(f"  TATR_INIT_MODE= {TATR_INIT_MODE}")
print(f"  INIT_SEG_MODE = {INIT_SEGMENT_MODE}")
print(f"  EVAL_YEARS    = {EVAL_YEARS}  (mutable; can be extended later)")
print(f"  SYN_MAX_YEARS = {SYN_MAX_YEARS}  (synthetic generated once per run)")
print(f"  SYN_CACHE_VER = {SYN_CACHE_VERSION}")
print(f"  Total LSTMs   = {N_RUNS * len(EVAL_YEARS)} (this session)")
print()
print(f"Drive paths:")
print(f"  ARCH_DIR = {ARCH_DIR}")
print(f"  SYN_DIR  = {SYN_DIR}")
print(f"  RES_DIR  = {RES_DIR}")
print(f"  FIG_DIR  = {FIG_DIR}")

## 3. Phase 1 — Train FTS-Diffusion Architecture (Once)

This step is **expensive (~1 hour on A100)** but only runs once. Checkpoints persist on Drive.

In [ ]:
# Sync persistent checkpoints to working dirs (so train_ftsdiffusion finds them)

def _is_for_asset(filename, asset, kind):
    """Return True if filename belongs to the given asset."""
    if kind == 'res':
        # SISC artifacts: sisc_{asset}_k{K}_l{lmin}-{lmax}_*.csv
        return filename.startswith(f'sisc_{asset}_') and filename.endswith('.csv')
    if kind == 'trained_models':
        # PGM/PEM filenames contain the asset name (e.g. pgm-2_c48-80_sp500_*)
        return (f'_{asset}_' in filename) and (filename.endswith('.pth') or filename.endswith('.pt'))
    if kind == 'data':
        return filename == f'{asset}_timeseries.csv'
    return False


def sync_persistent_to_working(persistent_dir, working_subdir, asset=None, kind=None):
    """Copy from Drive into working dir if working dir is empty (per file).
    Optionally filter by asset/kind to avoid cross-asset contamination."""
    src = persistent_dir
    dst = os.path.join(REPO_ROOT, working_subdir)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        if not os.path.exists(dst_f):
            shutil.copy(src_f, dst_f)
            print(f"  Restored {f}")


def sync_working_to_persistent(working_subdir, persistent_dir, asset=None, kind=None):
    """Copy from working dir to Drive (overwrite). Filter by asset/kind for clean separation."""
    src = os.path.join(REPO_ROOT, working_subdir)
    dst = persistent_dir
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        shutil.copy(src_f, dst_f)


# Restore architecture checkpoints if they exist on Drive (filtered by current ASSET)
print(f"Checking for existing {ASSET} architecture checkpoints on Drive...")
sync_persistent_to_working(f'{ARCH_DIR}/res',             'res',             asset=ASSET, kind='res')
sync_persistent_to_working(f'{ARCH_DIR}/trained_models',  'trained_models',  asset=ASSET, kind='trained_models')
sync_persistent_to_working(f'{ARCH_DIR}/data',            'data',            asset=ASSET, kind='data')

In [ ]:
# Step 1: Download asset data if not already present
ts_path = f'data/{DATANAME}_timeseries.csv'
if not os.path.exists(ts_path):
    print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
    get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
else:
    print(f"\u2713 Data already at {ts_path}")

fts = load_actual_fts(DATANAME).squeeze()
print(f"{PRETTY_NAME} series: {len(fts)} points, range [{fts.min():.4f}, {fts.max():.4f}]")

### Auto-fix: PEM device bug (apply BEFORE Phase 1 training)

The authors' `pattern_evolution_module.train_evolution_module` does not move target tensors to GPU before computing `cross_entropy`, causing a CUDA device-mismatch error on most modern PyTorch versions. This cell patches the source file in the cloned repo before any architecture training.

**Idempotent**: safe to re-run; does nothing if already patched. Keep this cell in the notebook permanently — it will be applied automatically every Colab session (since the patch lives on the cloned repo, which is reset on disconnect).

In [ ]:
# === AUTO-FIX: PEM target tensors not moved to GPU (idempotent) ===
import sys, importlib

pem_file = f'{REPO_ROOT}/models/pattern_evolution_module.py'
PATCH_MARKER = '_FTS_PATCH_DEVICE_FIX_'

with open(pem_file) as f:
    code = f.read()

if PATCH_MARKER in code:
    print("\u2713 PEM file already patched")
else:
    old_block = (
        "      target_pattern = batch_y[:, 0].long()\n"
        "      target_length = (batch_y[:, 1] - 10).long()\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1)"
    )
    new_block = (
        "      # " + PATCH_MARKER + "\n"
        "      _dev = next(model.parameters()).device\n"
        "      batch_x = batch_x.to(_dev)\n"
        "      target_pattern = batch_y[:, 0].long().to(_dev)\n"
        "      target_length = (batch_y[:, 1] - 10).long().to(_dev)\n"
        "      target_magnitude = batch_y[:, 2].float().view(-1, 1).to(_dev)"
    )
    if old_block not in code:
        print("\u26a0 Pattern not found \u2014 file may have changed. Manual inspection required.")
    else:
        code = code.replace(old_block, new_block)
        with open(pem_file, 'w') as f:
            f.write(code)
        print(f"\u2713 Patched {pem_file}")

# Verify patch is in file
with open(pem_file) as f:
    content = f.read()
assert PATCH_MARKER in content, "PATCH NOT APPLIED"

# Aggressive cache purge to ensure reloaded modules pick up the new code
mods_to_purge = [m for m in list(sys.modules.keys())
                 if any(p in m for p in [
                     'pattern_evolution_module',
                     'pattern_recognition_module',
                     'pattern_generation_module',
                     'pattern_conditioned_diffusion',
                     'scaling_autoencoder',
                     'train_ftsdiffusion',
                     'load_models',
                 ])]
for m in mods_to_purge:
    del sys.modules[m]

# Re-import fresh
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)

# Verify the loaded function contains the patch
import models.pattern_evolution_module as pem_mod
import inspect
src_code = inspect.getsource(pem_mod.train_evolution_module)
assert PATCH_MARKER in src_code, "Loaded function does NOT have patch \u2014 try Runtime > Restart runtime"

print("\u2713 Modules reloaded; patched function is active")

In [ ]:
# Step 2: Train SISC + PGM + PEM (only if no checkpoint for this asset)
has_all = (_has_recognition_artifacts() and
           _has_generation_artifact() and
           _has_evolution_artifact())

if has_all:
    print(f"\u2713 All {ASSET} architecture artifacts found \u2014 skipping training.")
else:
    print(f"Training FTS-Diffusion architecture for {PRETTY_NAME} (SISC + PGM + PEM)...")
    print("Estimated time: ~50-75 min on A100")
    t0 = time.time()
    train_ftsdiffusion(fts, train_test_split=0.8, store_model=True)
    elapsed = (time.time() - t0) / 60
    print(f"\u2713 Architecture training complete in {elapsed:.1f} min")

# Persist asset-specific checkpoints to Drive
print(f"Syncing {ASSET} checkpoints to Drive...")
sync_working_to_persistent('res', f'{ARCH_DIR}/res', asset=ASSET, kind='res')
sync_working_to_persistent('trained_models', f'{ARCH_DIR}/trained_models', asset=ASSET, kind='trained_models')
sync_working_to_persistent('data', f'{ARCH_DIR}/data', asset=ASSET, kind='data')
print(f"\u2713 {ASSET} checkpoints persisted on Drive at {ARCH_DIR}.")

## 3.5 Visualizza i pattern SISC appresi

Dopo Phase 1 (allenamento dell'architettura), SISC ha discovered K=14 pattern ricorrenti dalla serie storica. Visualizziamoli.

In [ ]:
# === SISC: visualizza i K pattern appresi ===
import pandas as pd
import matplotlib.pyplot as plt

K = prm_params['k']
sisc_prefix = f'res/sisc_{ASSET}_k{K}_l{prm_params["l_min"]}-{prm_params["l_max"]}_dba_kmpp'

centroids_path = f'{sisc_prefix}_centroids.csv'
labels_path = f'{sisc_prefix}_labels.csv'
segmentation_path = f'{sisc_prefix}_segmentation.csv'

if not all(os.path.exists(p) for p in [centroids_path, labels_path, segmentation_path]):
    print(f"\u26a0 SISC artifacts not found in {sisc_prefix}_*.csv")
    print(f"  Run Phase 1 first (or sync from Drive)")
else:
    centroids = pd.read_csv(centroids_path).values[:, 1:]   # (K, l_max)
    labels = pd.read_csv(labels_path).values[:, 1].astype(int)
    segmentation = pd.read_csv(segmentation_path).values[:, 1].astype(int)
    seg_lengths = np.diff(segmentation)

    # === 1) Plot: K pattern appresi ===
    n_cols = 5
    n_rows = (K + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
    axes = axes.flatten()

    # Color palette: each pattern gets its own color
    cmap = plt.cm.get_cmap('tab20', K)

    for k in range(K):
        ax = axes[k]
        ax.plot(centroids[k], linewidth=2, color=cmap(k))
        ax.set_title(f'Pattern {k}  (n={int(np.sum(labels == k))} segs)', fontsize=10)
        ax.grid(alpha=0.3)
        ax.set_xticks([])

    # Hide unused subplots
    for j in range(K, len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'SISC patterns learned for {PRETTY_NAME} (K={K})',
                 fontsize=13, fontweight='bold', y=1.005)
    plt.tight_layout()

    # Save
    sisc_fig_dir = f'{PERSIST_ROOT}/figures/sisc/{ASSET}'
    os.makedirs(sisc_fig_dir, exist_ok=True)
    fig.savefig(f'{sisc_fig_dir}/patterns.pdf', bbox_inches='tight', dpi=150)
    fig.savefig(f'{sisc_fig_dir}/patterns.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\u2713 Saved SISC patterns to {sisc_fig_dir}/patterns.{{pdf,png}}")

    # === 2) Statistiche di frequenza dei pattern ===
    print(f"\nPattern frequency in {PRETTY_NAME} segmentation:")
    print(f"  Total segments: {len(labels)}")
    print(f"  Total covered days: {seg_lengths.sum()}")
    print(f"  Mean segment length: {seg_lengths.mean():.2f} days")
    print(f"  Pattern occurrences: {np.bincount(labels, minlength=K).tolist()}")

    # === 3) Plot: distribuzione di frequenza ===
    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4))

    # Bar plot: frequenza dei pattern
    counts = np.bincount(labels, minlength=K)
    axes2[0].bar(range(K), counts, color=[cmap(k) for k in range(K)])
    axes2[0].set_xlabel('Pattern ID')
    axes2[0].set_ylabel('Number of segments')
    axes2[0].set_title(f'Pattern frequency ({PRETTY_NAME})')
    axes2[0].set_xticks(range(K))
    axes2[0].grid(alpha=0.3, axis='y')

    # Hist: distribuzione delle lunghezze dei segmenti
    axes2[1].hist(seg_lengths, bins=range(int(seg_lengths.min()), int(seg_lengths.max())+2),
                  edgecolor='black', alpha=0.7, color='steelblue')
    axes2[1].set_xlabel('Segment length (days)')
    axes2[1].set_ylabel('Frequency')
    axes2[1].set_title(f'Segment length distribution ({PRETTY_NAME})')
    axes2[1].grid(alpha=0.3, axis='y')

    plt.tight_layout()
    fig2.savefig(f'{sisc_fig_dir}/pattern_stats.pdf', bbox_inches='tight', dpi=150)
    fig2.savefig(f'{sisc_fig_dir}/pattern_stats.png', bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig2)
    print(f"\u2713 Saved SISC stats to {sisc_fig_dir}/pattern_stats.{{pdf,png}}")

## 4. Phase 2 — TATR Loop

For each of the 15 runs:
1. Generate `MAX_YEARS × 252 = 25,200` days of synthetic data (~12 min on A100)
2. For each evaluation year in `[0, 10, 20, ..., 100]`:
   - Build augmented dataset = real init + synthetic [0:year × 252]
   - Train LSTM (full-batch, 100 epochs, lr=1e-2, MAE loss, hidden=32)
   - Test on real test set → record MAPE
3. Save run results to CSV after each run.

In [ ]:
# === Setup TATR datasets ===
# S&P 500 uses the exact reference-code split: 252*5 held-out days for initial
# LSTM training, then the rest of the held-out block for evaluation.
# Shorter assets use an explicitly labeled adaptive fallback, not a paper-exact claim.

from experiments.utils_downstream import (
    get_downstream_data, init_tatr_set,
    Timeseries2Dataset_Downstream
)


def init_observed_segment(segments_test, labels_test, lengths_test, init_period, seed=None):
    """Choose the synthetic-series initial segment from observed TATR init data."""
    if INIT_SEGMENT_MODE == 'reference_first':
        idx = 0
    elif INIT_SEGMENT_MODE == 'paper_sampled':
        # Restrict sampling to segments fully inside the limited observed init set,
        # so synthetic generation is not initialized from the real evaluation period.
        starts = np.r_[0, np.cumsum(lengths_test[:-1])]
        ends = starts + lengths_test
        candidates = np.where(ends <= init_period)[0]
        if len(candidates) == 0:
            raise ValueError('No complete observed segment is available inside the TATR init period.')
        rng = np.random.RandomState(seed) if seed is not None else np.random
        idx = int(rng.choice(candidates))
    else:
        raise ValueError(f'Unknown INIT_SEGMENT_MODE={INIT_SEGMENT_MODE!r}')

    init_segment = segments_test[idx]
    init_pattern = labels_test[idx]
    init_length = lengths_test[idx]
    init_magnitude = max(init_segment) - min(init_segment)
    init_state = np.stack((init_pattern, init_length, init_magnitude), axis=0)
    init_state = torch.tensor(init_state).view(1, -1)
    return (init_state, init_segment), idx


def setup_dowmstream_tatr_reproduction(window_size, init_period_days=252*5,
                                       fallback_fraction=0.625, min_test_days=252):
    global TATR_INIT_SEGMENT_POOL
    downstream_timeseries, segments_test, labels_test, lengths_test = get_downstream_data()
    total_len = len(downstream_timeseries)

    if TATR_INIT_MODE == 'paper':
        init_period = init_period_days
        if total_len <= init_period + window_size:
            raise ValueError(
                f'Paper TATR split needs more than {init_period + window_size} held-out days; '
                f'got {total_len}. Use ASSET="sp500" or set TATR_INIT_MODE="adaptive".'
            )
    elif TATR_INIT_MODE == 'adaptive':
        max_init = total_len - min_test_days
        init_period = min(int(total_len * fallback_fraction), max_init)
        init_period = max(init_period, min_test_days)
    else:
        raise ValueError(f'Unknown TATR_INIT_MODE={TATR_INIT_MODE!r}')

    init_timeseries = downstream_timeseries[:init_period]
    test_timeseries = downstream_timeseries[init_period:]

    TATR_INIT_SEGMENT_POOL = (segments_test, labels_test, lengths_test, init_period)
    init_dataset, scaler = init_tatr_set(init_timeseries, window_size)
    first_segment, first_segment_idx = init_observed_segment(
        segments_test, labels_test, lengths_test, init_period, seed=42)
    test_dataset = init_tatr_set(test_timeseries, window_size, scaler)

    print(f"TATR setup for {ASSET} ({PRETTY_NAME}):")
    print(f"  mode:                        {TATR_INIT_MODE}")
    print(f"  init segment mode:           {INIT_SEGMENT_MODE}")
    print(f"  init segment index:          {first_segment_idx}")
    print(f"  Total downstream timeseries: {total_len} days ({total_len/252:.2f} years)")
    print(f"  Init period (LSTM training): {init_period} days ({init_period/252:.2f} years)")
    print(f"  Test period (LSTM eval):     {total_len - init_period} days ({(total_len-init_period)/252:.2f} years)")
    print(f"  init_dataset shape:  {init_dataset.shape}")
    print(f"  test_dataset shape:  {test_dataset.shape}")
    return init_dataset, first_segment, test_dataset, scaler


init_dataset, first_segment, test_dataset, scaler = setup_dowmstream_tatr_reproduction(WINDOW_SIZE)
init_state, init_segment = first_segment

In [ ]:
# === LSTM training & evaluation functions (paper-faithful) ===

def train_lstm_full_batch(dataset, n_epochs=N_EPOCHS, hidden_dim=LSTM_HIDDEN,
                          n_layers=LSTM_LAYERS, output_dim=AHEAD, criterion_str=LSTM_LOSS,
                          lr=LR, seed=0):
    """Match the authors' full-batch training in predictor_lstm.py exactly."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=output_dim,
                          n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    X = dataset[:, :-output_dim].unsqueeze(-1).to(device)
    y = dataset[:, -output_dim:].to(device)
    
    for epoch in range(n_epochs):
        model.train()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return model, float(loss.item())


def evaluate_lstm(model, test_dataset, scaler, output_dim=AHEAD):
    """Compute MAPE on the real test set, on the original (un-scaled) scale."""
    model.eval()
    with torch.no_grad():
        X = test_dataset[:, :-output_dim].unsqueeze(-1).to(device)
        y_true = test_dataset[:, -output_dim:].numpy()
        y_pred = model(X).cpu().numpy()
        # Inverse-transform back to original scale
        y_true_orig = scaler.inverse_transform(y_true)
        y_pred_orig = scaler.inverse_transform(y_pred)
    return float(MAPE(y_true_orig, y_pred_orig))

print("\u2713 Train/eval functions ready")

In [ ]:
# === TATR run for a single seed (AUTHORS' PROTOCOL: independent 252-day blocks) ===

def run_single_tatr(run_idx, eval_years, seed_base=42):
    """Run one TATR replicate following the authors' protocol exactly.

    Authors' protocol (matches reference code in experiments/tatr.py):
      - At each augmentation step k, generate a FRESH 252-day synthetic block.
      - Initial segment selection is controlled by INIT_SEGMENT_MODE:
        paper_sampled samples from observed TATR init data; reference_first uses segment 0.
      - Concatenate this block (after unfolding to windows) with the cumulative
        training dataset.
      - Each call to generate_timeseries_ftsdiffusion is stochastically independent.
      - Boundary effect: each 252-day block produces 189 windows. Concatenating
        100 blocks gives 18,900 windows (vs 25,137 for one continuous trajectory).
        This is faithful to the paper's data structure.

    Resumability:
      - Per-block caching: blocks are stored as a single .npy file of shape
        [n_blocks, 252]. Already-generated blocks are reused; new blocks are
        generated only when needed.
      - Per-year MAPE caching: results are saved to CSV after each year.

    Adding new evaluation years later:
      - Append more years to EVAL_YEARS in the config cell.
      - This function will load all previously-saved years (skip them).
      - For new years, it loads the cached blocks (no regeneration) and computes
        the missing MAPE values.
    """
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    blocks_path = f'{SYN_DIR}/run_{run_idx:02d}_{TATR_INIT_MODE}_{INIT_SEGMENT_MODE}_{SYN_CACHE_VERSION}_blocks.npy'

    # === Load any pre-existing (possibly partial) MAPE results ===
    results = {}
    if os.path.exists(run_csv):
        df_prev = pd.read_csv(run_csv)
        results = dict(zip(df_prev['eval_year'].astype(int), df_prev['mape'].astype(float)))
        if set(eval_years).issubset(set(results.keys())):
            print(f"  \u2713 Run {run_idx}: all requested years already in CSV \u2014 skipping.")
            return {ey: results[ey] for ey in eval_years}
        elif results:
            done = sorted(results.keys())
            missing = sorted(set(eval_years) - set(done))
            print(f"  \u26a0 Run {run_idx} partial: years already saved={done}, to compute={missing}")

    np.random.seed(seed_base + run_idx)
    torch.manual_seed(seed_base + run_idx)

    # === Per-block synthetic cache: load existing blocks, generate missing ones ===
    # Generate the configured maximum once per run so adding EVAL_YEARS later cannot
    # append blocks from a reset RNG stream. Each block also gets its own deterministic
    # seed, making interrupted/resumed generation stable.
    n_blocks_needed = SYN_MAX_YEARS
    cached_blocks = None
    if os.path.exists(blocks_path):
        cached_blocks = np.load(blocks_path)
        n_existing = cached_blocks.shape[0] if cached_blocks.ndim == 2 else 0
        print(f"  Loading {n_existing} cached synthetic blocks for run {run_idx}")
    else:
        n_existing = 0

    if n_existing < n_blocks_needed:
        n_to_generate = n_blocks_needed - n_existing
        print(f"  Generating {n_to_generate} new 252-day synthetic blocks (~{n_to_generate*0.05:.1f} min on A100)...")
        t0 = time.time()
        new_blocks = []
        for b in range(n_existing, n_blocks_needed):
            block_seed = seed_base + run_idx * (SYN_MAX_YEARS + 1) + b
            np.random.seed(block_seed)
            torch.manual_seed(block_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(block_seed)

            block_first_segment, block_first_segment_idx = init_observed_segment(
                *TATR_INIT_SEGMENT_POOL, seed=block_seed)
            block_init_state, block_init_segment = block_first_segment

            # Each block: fresh 252-day generation from the configured observed initial segment.
            block = generate_timeseries_ftsdiffusion(
                T=AUG_LENGTH,
                init_state=block_init_state,
                init_segment=block_init_segment)
            # Ensure exactly AUG_LENGTH points
            block = np.asarray(block, dtype=np.float32)[:AUG_LENGTH]
            if len(block) < AUG_LENGTH:
                # Pad if generation returned slightly less (rare edge case)
                block = np.concatenate([block, np.full(AUG_LENGTH - len(block), block[-1] if len(block) > 0 else 0.0)])
            new_blocks.append(block)

            # Save incrementally every 10 blocks (in case we get interrupted)
            if (b + 1) % 10 == 0:
                stacked_so_far = np.stack(new_blocks, axis=0)
                if cached_blocks is not None:
                    full_so_far = np.concatenate([cached_blocks, stacked_so_far], axis=0)
                else:
                    full_so_far = stacked_so_far
                np.save(blocks_path, full_so_far)

        new_blocks_arr = np.stack(new_blocks, axis=0)
        if cached_blocks is not None:
            all_blocks = np.concatenate([cached_blocks, new_blocks_arr], axis=0)
        else:
            all_blocks = new_blocks_arr
        np.save(blocks_path, all_blocks)
        print(f"  Block generation: {(time.time()-t0)/60:.1f} min \u2014 {n_to_generate} blocks saved")
    else:
        all_blocks = cached_blocks

    print(f"  Total blocks available: {all_blocks.shape[0]} (need {n_blocks_needed})")

    # === Per-year LSTM evaluation with INCREMENTAL save ===
    for ey in eval_years:
        if ey in results:
            print(f"  Run {run_idx:2d} | Year {ey:3d} | already done \u2014 MAPE={results[ey]:.5f} (loaded)")
            continue

        # Build augmented dataset following AUTHORS' protocol:
        # init_dataset + concat of each block's unfolded windows
        if ey == 0:
            curr_dataset = copy_dataset_downstream(init_dataset)
        else:
            curr_dataset = copy_dataset_downstream(init_dataset)
            for b in range(ey):
                block = all_blocks[b]  # 252-day 1D series
                # create_syn_dataset = scale + unfold internally
                block_dataset = create_syn_dataset(block, WINDOW_SIZE, scaler, DATATYPE)
                curr_dataset = concat_datasets_downstream(curr_dataset, block_dataset)

        # Train + eval
        t0 = time.time()
        lstm_seed = seed_base + 100_000 + run_idx * (SYN_MAX_YEARS + 1) + ey
        model, train_loss = train_lstm_full_batch(curr_dataset, seed=lstm_seed)
        train_time = time.time() - t0
        mape = evaluate_lstm(model, test_dataset, scaler)
        results[ey] = mape

        print(f"  Run {run_idx:2d} | Year {ey:3d} | windows={curr_dataset.shape[0]:6d} | "
              f"train_loss={train_loss:.4f} | MAPE={mape:.5f} | LSTM={train_time:.1f}s")

        # === SAVE TO DRIVE IMMEDIATELY ===
        sorted_years = sorted(results.keys())
        df_save = pd.DataFrame({
            'eval_year': sorted_years,
            'mape': [results[y] for y in sorted_years]
        })
        df_save.to_csv(run_csv, index=False)

        # Free GPU memory
        del model, curr_dataset
        torch.cuda.empty_cache()

    return {ey: results[ey] for ey in eval_years if ey in results}

print("\u2713 TATR runner ready (AUTHORS' PROTOCOL: independent blocks, per-block caching)")

In [ ]:
# ============================================================================
# === MAIN LOOP \u2014 runs all replicates with live plotting + mean tracking ===
# ============================================================================
%matplotlib inline
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

all_results = {}
running_mape = np.full((N_RUNS, len(EVAL_YEARS)), np.nan)

# === STEP 1: Pre-load any runs already saved on Drive ===
print(f"Pre-loading existing runs from {RES_DIR}...")
for run_idx in range(N_RUNS):
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if os.path.exists(run_csv):
        df = pd.read_csv(run_csv)
        for j, ey in enumerate(EVAL_YEARS):
            mask = df['eval_year'] == ey
            if mask.any():
                running_mape[run_idx, j] = df.loc[mask, 'mape'].values[0]
        all_results[run_idx] = dict(zip(df['eval_year'], df['mape']))
n_preloaded = int(sum(~np.all(np.isnan(running_mape), axis=1)))
print(f"  \u2713 Found {n_preloaded} existing runs on Drive\n")


# === STEP 2: Mean tracker ===
mean_history = []

def update_mean_history():
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return None
    means = np.nanmean(running_mape[valid], axis=0)
    snap = {'n_runs': n_done}
    for j, ey in enumerate(EVAL_YEARS):
        snap[f'mean_y{ey}'] = float(means[j]) if not np.isnan(means[j]) else None
    mean_history.append(snap)
    return means


# === STEP 3: Live plot helper ===
def plot_live(elapsed_min=None):
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    for r in range(N_RUNS):
        if not valid[r]:
            continue
        # Skip NaN points (years not yet computed for this run)
        mask = ~np.isnan(running_mape[r])
        ax.plot(np.array(EVAL_YEARS)[mask], running_mape[r][mask],
                color='steelblue', alpha=0.3, linewidth=1)

    means = np.nanmean(running_mape[valid], axis=0) if n_done >= 1 else np.full(len(EVAL_YEARS), np.nan)
    if n_done >= 2:
        stds = np.nanstd(running_mape[valid], axis=0)
        ax.fill_between(EVAL_YEARS, means - stds, means + stds,
                        alpha=0.25, color='steelblue', label=f'\u00b11 std (n={n_done})')
    valid_mean = ~np.isnan(means)
    ax.plot(np.array(EVAL_YEARS)[valid_mean], means[valid_mean],
            'o-', linewidth=2.5, color='darkblue', markersize=8, label='Mean MAPE')
    if not np.isnan(means[0]):
        ax.axhline(means[0], color='gray', linestyle='--', alpha=0.7,
                   label=f'Year-0 baseline = {means[0]:.4f}')

    for x, y in zip(EVAL_YEARS, means):
        if np.isnan(y):
            continue
        ax.annotate(f'{y:.4f}', xy=(x, y), xytext=(0, 12),
                    textcoords='offset points', ha='center',
                    fontsize=9, fontweight='bold', color='darkblue',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='darkblue', alpha=0.85, linewidth=0.8))

    title = f'{EXPERIMENT.upper()} live \u2014 {PRETTY_NAME} \u2014 {n_done}/{N_RUNS} runs done'
    if elapsed_min is not None:
        title += f' (last: {elapsed_min:.1f} min)'
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Augmentation (years)', fontsize=11)
    ax.set_ylabel('MAPE on real test set', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax + 0.015)
    plt.tight_layout()

    # Save current state to Drive (OVERWRITES — single live snapshot per asset+experiment)
    fig.savefig(f'{FIG_DIR}/live.png', dpi=120, bbox_inches='tight')

    display(fig)
    plt.close(fig)


# === STEP 4: Mean delta printer ===
def print_mean_delta(prev_means, new_means, n_done):
    if prev_means is None or new_means is None:
        if new_means is not None:
            print(f"\n[Mean tracking] First run completed (n=1). Initial means:")
            for ey, m in zip(EVAL_YEARS, new_means):
                if not np.isnan(m):
                    print(f"  Year {ey:3d}: {m:.5f}")
        return
    print(f"\n[Mean tracking] After run {n_done} (n={n_done}):")
    print(f"  {'Year':>4} | {'Old mean':>10} | {'New mean':>10} | {'\u0394':>10}")
    print(f"  {'-'*4}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}")
    for ey, old, new in zip(EVAL_YEARS, prev_means, new_means):
        if np.isnan(old) and np.isnan(new):
            print(f"  {ey:>4d} | {'NaN':>10} | {'NaN':>10} | {'\u2014':>10}")
            continue
        if np.isnan(old):
            print(f"  {ey:>4d} | {'NaN':>10} | {new:>10.5f} | NEW")
            continue
        if np.isnan(new):
            print(f"  {ey:>4d} | {old:>10.5f} | {'NaN':>10} | {'\u2014':>10}")
            continue
        delta = new - old
        sign = '\u2193' if delta < 0 else ('\u2191' if delta > 0 else '=')
        print(f"  {ey:>4d} | {old:>10.5f} | {new:>10.5f} | {sign} {abs(delta):.5f}")


# === STEP 5: Initialize history with pre-loaded data ===
prev_means = update_mean_history()
plot_live()


# === STEP 6: Main run loop ===
for run_idx in range(N_RUNS):
    print(f"\n{'='*70}")
    print(f"Run {run_idx + 1}/{N_RUNS} \u2014 {PRETTY_NAME} ({EXPERIMENT.upper()})")
    print(f"{'='*70}")

    t0 = time.time()
    results = run_single_tatr(run_idx, EVAL_YEARS)
    all_results[run_idx] = results
    elapsed = (time.time() - t0) / 60

    # === FIX: handle missing years gracefully (no KeyError) ===
    for j, ey in enumerate(EVAL_YEARS):
        running_mape[run_idx, j] = results.get(ey, np.nan)

    new_means = update_mean_history()
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    print_mean_delta(prev_means, new_means, n_done)
    prev_means = new_means.copy() if new_means is not None else None

    plot_live(elapsed_min=elapsed)
    pd.DataFrame(mean_history).to_csv(f'{RES_DIR}/mean_history.csv', index=False)

    print(f"\u2713 Run {run_idx + 1} done in {elapsed:.1f} min")

print(f"\n{'='*70}\nALL {N_RUNS} RUNS COMPLETE for {PRETTY_NAME} ({EXPERIMENT.upper()})\n{'='*70}")

## 5. Phase 3 — Aggregation & Confidence Intervals

In [ ]:
# Reload all per-run CSVs from Drive (handles re-runs, partial completion)
rows = []
for run_idx in range(N_RUNS):
    p = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if not os.path.exists(p):
        print(f"\u26a0 Missing run {run_idx}")
        continue
    df = pd.read_csv(p)
    df['run_idx'] = run_idx
    rows.append(df)

df_all = pd.concat(rows, ignore_index=True)
df_pivot = df_all.pivot(index='run_idx', columns='eval_year', values='mape')
print(f"Results matrix: {df_pivot.shape} (runs \u00d7 eval_years)")
print("\nMAPE per run \u00d7 evaluation year:")
print(df_pivot.round(5))

In [ ]:
# Compute statistics: mean, 95% CI via percentile bootstrap
from scipy import stats as sstats

def bootstrap_ci(x, n_boot=10000, ci=0.95, seed=0):
    """Percentile bootstrap CI for the mean."""
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [(1-ci)/2, 1-(1-ci)/2])
    return lo, hi

summary_stats = []
for ey in EVAL_YEARS:
    vals = df_pivot[ey].dropna().values
    mean = vals.mean()
    std = vals.std()
    lo, hi = bootstrap_ci(vals)
    summary_stats.append({
        'eval_year': ey,
        'mean_mape': mean,
        'std_mape': std,
        'ci95_low': lo,
        'ci95_high': hi,
        'n_runs': len(vals)
    })

summary_df = pd.DataFrame(summary_stats)
print("\nSummary statistics (95% bootstrap CI over runs):")
print(summary_df.round(5))

# Save
summary_df.to_csv(f'{RES_DIR}/summary.csv', index=False)
df_pivot.to_csv(f'{RES_DIR}/results_matrix.csv')
print(f"\n\u2713 Saved to {RES_DIR}/summary.csv and results_matrix.csv")

## 6. Final Figure — Replicate Fig. 6(b) of the Paper

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

x = summary_df['eval_year'].values * AUG_LENGTH
mean = summary_df['mean_mape'].values
lo = summary_df['ci95_low'].values
hi = summary_df['ci95_high'].values

ax.plot(x, mean, 'o-', linewidth=2.5, color='steelblue', markersize=7,
        label=f'FTS-Diffusion (n={N_RUNS} runs)')
ax.fill_between(x, lo, hi, alpha=0.25, color='steelblue', label='95% bootstrap CI')
ax.axhline(mean[0], color='gray', linestyle='--', alpha=0.7,
           label=f'Initial MAPE (year 0) = {mean[0]:.4f}')

# Annotate mean on each marker
for xv, yv in zip(x, mean):
    ax.annotate(f'{yv:.4f}', xy=(xv, yv), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=9,
                fontweight='bold', color='darkblue',
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='darkblue', alpha=0.85, linewidth=0.8))

if EVAL_YEARS[-1] == 100:
    pct_change = 100 * (mean[-1] - mean[0]) / mean[0]
    ax.annotate(f'\u0394MAPE @ year 100: {pct_change:+.1f}%\n(paper: -17.9%)',
                xy=(x[-1], mean[-1]), xytext=(x[-1]*0.55, mean[-1] + 0.02),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=11, color='red')

ax.set_xlabel('Augmented Size (days)', fontsize=12)
ax.set_ylabel('MAPE (real test set)', fontsize=12)
ax.set_title(f'{EXPERIMENT.upper()} \u2014 {PRETTY_NAME} \u2014 {N_RUNS} runs \u00d7 {len(EVAL_YEARS)} eval points '
             f'= {N_RUNS*len(EVAL_YEARS)} LSTMs', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 0.015)
plt.tight_layout()

fig_path = f'{FIG_DIR}/final.pdf'
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.savefig(fig_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
plt.show()
print(f"\u2713 Figure saved to {fig_path}")

## 7. Quality Checks

In [ ]:
# === Sanity checks ===
print("\n" + "="*70)
print("SANITY CHECKS")
print("="*70)

# 1) Every run completed?
print(f"\n[1] Runs completed: {len(df_pivot)}/{N_RUNS}")

# 2) Per-eval-year counts (should all be N_RUNS)
print(f"\n[2] Run counts per eval year:")
print(df_pivot.notna().sum(axis=0).to_string())

# 3) Trend: mean MAPE should generally DECREASE with more synthetic data (paper claim)
print(f"\n[3] Trend check — mean MAPE per eval year:")
print(summary_df[['eval_year', 'mean_mape']].to_string(index=False))
trend = np.polyfit(summary_df['eval_year'], summary_df['mean_mape'], 1)
print(f"  Linear trend slope: {trend[0]:.2e} ({'\u2193 downward (good)' if trend[0] < 0 else '\u2191 upward (NOT matching paper)'})")

# 4) Variance: across-run std should be reasonable (not all runs collapsing)
print(f"\n[4] Across-run std at each eval year:")
print(summary_df[['eval_year', 'std_mape']].to_string(index=False))

# 5) Year 0 baseline — paper reports ~0.026-0.05 MAPE
y0_mean = summary_df.iloc[0]['mean_mape']
print(f"\n[5] Year-0 baseline MAPE: {y0_mean:.5f} (paper Fig.6(b) starts ~0.05)")

# 6) Compute % reduction at year 100
if EVAL_YEARS[-1] == 100:
    y100_mean = summary_df.iloc[-1]['mean_mape']
    pct = 100 * (y0_mean - y100_mean) / y0_mean
    print(f"\n[6] MAPE reduction year 0 \u2192 100: {pct:+.2f}% (paper: 17.9%)")

## 8. How to Resume / Switch Asset / Add TMTR

### Resume after Colab disconnect
1. Reopen this notebook
2. Re-run cells 1-3 (setup, imports, config)
3. Re-run Phase 1 — it detects existing checkpoints for the current ASSET on Drive and skips
4. Re-run the main TATR loop — it detects completed runs in `RES_DIR` and skips them

### Switch to a different asset
1. Change `ASSET = 'goog'` (or `'zcf'`) in the **Configuration** cell
2. Re-run from Phase 1 onwards
3. Phase 1 will train the architecture for the new asset (~1h on A100)
4. Phase 2 will run the experiment for that asset
5. Results stored separately under `{PERSIST_ROOT}/results/{exp}/{ASSET}/`

### Switch from TATR to TMTR
1. Change `EXPERIMENT = 'tmtr'` in Configuration
2. (You will need a TMTR-specific main loop \u2014 currently this notebook implements TATR. The folder structure is ready for TMTR results to be stored at `{PERSIST_ROOT}/results/tmtr/{ASSET}/`.)

### Drive storage layout
```
/content/drive/MyDrive/fts_diffusion/
  \u251c\u2500\u2500 architectures/{asset}/   \u2190 SISC + PGM + PEM checkpoints (per asset)
  \u251c\u2500\u2500 synthetic/{asset}/        \u2190 Per-run synthetic series .npy
  \u251c\u2500\u2500 results/{exp}/{asset}/    \u2190 Per-run MAPE CSVs + summary + mean_history
  \u2514\u2500\u2500 figures/{exp}/            \u2190 Final plots
```

## Caveats / Sources of Variability

Even with `torch.manual_seed`, exact reproducibility on different GPUs is hard:
- **Different GPU = different cuDNN kernels** \u2192 small numerical differences
- **Sampling pipeline uses randomness in the diffusion noise** \u2192 different seed = different synthetic series
- **Across-run variance** is the *intended* signal: it captures uncertainty from generative randomness

If your year-100 MAPE reduction is in the range -10% to -25%, you've replicated the paper qualitatively.